<a href="https://colab.research.google.com/github/cassiecinzori/ECON3916/blob/main/Labs/Lecture11/Dirty_Data_Forensics_and_Structural_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dirty Data Forensics and Structural Engineering

### Cassandra Cinzori

### Step 1: Environment Initialization and Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import missingno as msno
import category_encoders as ce
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/cassiecinzori/reference_data/refs/heads/main/messy_hr_economics.csv'
df = pd.read_csv(url)

### Step 2: Visual Forensics of Missing Data

In [ ]:
msno.matrix(df)
plt.title("Missingness Matrix – look for aligned white stripes")
plt.tight_layout()
plt.show()

### Step 3: Handling the Missingness via Conditional Imputation

In [ ]:
df['base_salary'] = (
    df.groupby('department')['base_salary']
      .transform(lambda x: x.fillna(x.median()))
)

### Step 4: Springing the Dummy Variable Trap

In [ ]:
# Step 4: The Dummy Variable Trap (Intentional Failure)
dummies_trap = pd.get_dummies(df['department']).astype(int)

X_trap = pd.concat([df[['tenure_years']], dummies_trap], axis=1)
X_trap = sm.add_constant(X_trap)
y = df['base_salary']

model_trap = sm.OLS(y, X_trap).fit()
print(model_trap.summary())

### Step 5: Escaping the Trap and Executing Advanced Encoding

In [ ]:
dummies_safe = pd.get_dummies(df['department'], drop_first=True).astype(int)

X_safe = pd.concat([df[['tenure_years']], dummies_safe], axis=1)
X_safe = sm.add_constant(X_safe)

model_safe = sm.OLS(y, X_safe).fit()
print(model_safe.summary())


encoder = ce.TargetEncoder(cols=['office_zip'])
df['zip_encoded'] = encoder.fit_transform(df['office_zip'], df['base_salary'])

print(df[['office_zip', 'zip_encoded']].head(10))

### AI Expansion

In [ ]:
# AI Expansion: Interactive Visual Forensics Dashboard (Colab-compatible)
# Uses ipywidgets instead of Streamlit — runs natively in Colab/Jupyter.
#
# ipywidgets provides interactive UI elements (dropdowns, buttons, output areas)
# that render inline in the notebook. No separate server needed.
# The dashboard state (imputed df) is held in a plain Python dict called
# `state`, which persists across widget callbacks for the lifetime of the cell.

import ipywidgets as widgets
from IPython.display import display, clear_output
import missingno as msno
import matplotlib.pyplot as plt
import pandas as pd

# state dict holds the working DataFrame so imputation changes persist
# across button clicks without re-running the whole cell
state = {'df': df.copy()}

# ------------------------------------------------------------------
# SECTION 1: Null audit output (renders immediately on run)
# ------------------------------------------------------------------
print("=" * 50)
print("VISUAL FORENSICS DASHBOARD")
print("=" * 50)

print("\n📋 Data Preview (first 5 rows):")
display(state['df'].head())

print("\n🔍 Null Value Counts:")
null_counts = state['df'].isnull().sum().rename("missing").to_frame()
null_counts["% missing"] = (null_counts["missing"] / len(state['df']) * 100).round(2)
display(null_counts[null_counts["missing"] > 0])

# ------------------------------------------------------------------
# SECTION 2: Missingness matrix button
# ------------------------------------------------------------------
btn_matrix = widgets.Button(
    description="Show Missingness Matrix",
    button_style="info",
    layout=widgets.Layout(width="220px")
)
matrix_out = widgets.Output()

def show_matrix(_):
    # widgets.Output() captures matplotlib figures and displays them
    # inline — without this, plt.show() would render outside the widget
    with matrix_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4))
        msno.matrix(state['df'], ax=ax, sparkline=False)
        ax.set_title("Missingness Matrix – aligned white stripes indicate MAR")
        plt.tight_layout()
        plt.show()

btn_matrix.on_click(show_matrix)
display(widgets.VBox([btn_matrix, matrix_out]))

# ------------------------------------------------------------------
# SECTION 3: Conditional imputation controls
# ------------------------------------------------------------------
print("\n⚙️ Conditional Median Imputation")

num_cols = state['df'].select_dtypes(include="number").columns.tolist()
cat_cols = state['df'].select_dtypes(include="object").columns.tolist()

# Dropdowns let the user select which column to impute and which to group by
dd_target = widgets.Dropdown(options=num_cols, description="Impute:")
dd_group  = widgets.Dropdown(options=cat_cols, description="Group by:")
btn_impute = widgets.Button(
    description="Run Imputation",
    button_style="success",
    layout=widgets.Layout(width="160px")
)
impute_out = widgets.Output()

def run_imputation(_):
    with impute_out:
        clear_output(wait=True)
        target = dd_target.value
        group  = dd_group.value
        before = state['df'][target].isnull().sum()

        # Update the shared state dict so the imputed df persists
        state['df'][target] = (
            state['df']
              .groupby(group)[target]
              .transform(lambda x: x.fillna(x.median()))
        )
        after = state['df'][target].isnull().sum()
        print(f"✅ Imputed {before - after} missing values in '{target}' "
              f"using per-'{group}' medians. Remaining nulls: {after}.")
        display(state['df'].head(10))

btn_impute.on_click(run_imputation)
display(widgets.VBox([
    widgets.HBox([dd_target, dd_group]),
    btn_impute,
    impute_out
]))